# Fresh NHAMCS data-to-simulation handoff
This notebook inspects the external dataset, runs the standalone exporter and tests, and reconciles the exported counts. No prior implementation is imported. Execution evidence does not determine competition eligibility. Run from the event_nhamcs directory using the intended Python environment.

In [1]:
import os, sys, platform, json, hashlib, subprocess
from pathlib import Path
import pandas as pd
print({"architecture": platform.machine(), "python": platform.python_version(), "pandas": pd.__version__})
print("Expected LinuxONE environment:", sys.executable.startswith("/home/linux1/jupyter-env/"))
root = Path.cwd()
assert (root / "export_parameters.py").is_file(), "Run from event_nhamcs"
remote_input = root.parent / "nhamcs_data_2018_22.csv"
local_input = root.parent.parent / "ed-surge-simulator/data/raw/nhamcs/nhamcs_data_2018_22.csv"
source = Path(os.environ.get("NHAMCS_INPUT", str(remote_input if remote_input.exists() else local_input)))
assert source.is_file()
source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
print({"input_file": source.name, "sha256": source_hash})

{'architecture': 's390x', 'python': '3.10.12', 'pandas': '2.3.3'}
Expected LinuxONE environment: True
{'input_file': 'nhamcs_data_2018_22.csv', 'sha256': '439a87b766cba5e8c7b838372f45b680ba29295b148078b50e8ab7056d20349b'}


## Inspect before transforming
Pandas inferred types are descriptive. The exporter parses original CSV tokens and does not use inferred floats for HHMM. No clinical field cleaning is performed.

In [2]:
raw = pd.read_csv(source, dtype=str, keep_default_na=False)
inferred = pd.read_csv(source)
print("Shape:", raw.shape)
print(pd.DataFrame({"inferred_dtype": inferred.dtypes.astype(str), "blank_cells": raw.apply(lambda s: s.str.strip().eq("").sum())}).to_string())
print("Exact duplicate excess rows:", int(raw.duplicated().sum()))
print("Arrival sample tokens:", raw["arrival_time"].head(10).tolist())

Shape: (58124, 49)


                       inferred_dtype  blank_cells
visit_month                     int64            0
day_of_week                     int64            0
arrival_time                  float64         1011
ems_arrival                    object            0
vitals_during_visit           float64          358
age                             int64            0
residence                      object            0
sex                            object            0
insurance                      object            0
no_payment                      int64            0
region                         object            0
temp                          float64         2045
heart_rate                    float64         1649
resp_rate                     float64         1582
sys_bp                        float64         5152
dias_bp                       float64         5199
spo2                          float64         2202
pain_score                    float64        14805
seen_last_72h                 f

## Export and independently validate
The CLI runs fresh synthetic tests and independent raw-data reconciliation, then records sanitized evidence. Reference checking is bound to the known source SHA-256.

In [3]:
command = [sys.executable, "-B", "export_parameters.py", "--input", str(source), "--output", "outputs/params.json", "--verify-reference", "--evidence"]
completed = subprocess.run(command, capture_output=True, text=True, check=True)
print(completed.stdout)
params = json.loads(Path("outputs/params.json").read_text())
evidence = json.loads(Path("outputs/execution_evidence.json").read_text())
assert evidence["source_unchanged"] and evidence["repeat_export_identical"]
assert all(evidence["reference_checks"].values())
assert hashlib.sha256(source.read_bytes()).hexdigest() == source_hash
print(json.dumps(evidence, indent=2))

{"records": 58124, "valid_arrivals": 57113, "valid_triage": 58124, "output_sha256": "f3f533d682665116a1a4d27aecbc2cd94121394bc42c15c2ad9ed867fbf42369"}

{
  "started_utc": "2026-10-17T14:18:17.902384+00:00",
  "completed_utc": "2026-10-17T14:18:19.924549+00:00",
  "system": "Linux",
  "architecture": "s390x",
  "uname_m": "s390x",
  "python": "3.10.12",
  "source_sha256": "439a87b766cba5e8c7b838372f45b680ba29295b148078b50e8ab7056d20349b",
  "output_sha256": "f3f533d682665116a1a4d27aecbc2cd94121394bc42c15c2ad9ed867fbf42369",
  "source_unchanged": true,
  "repeat_export_identical": true,
  "reference_checks": {
    "shape_58124_by_49": true,
    "arrival_denominator": true,
    "arrival_exclusions": true,
    "all_hour_counts": true,
    "peak_hour": true,
    "triage_denominator": true,
    "all_triage_counts": true,
    "probabilities": true
  },
  "tests": {
    "exit_code": 0,
    "summary": [
      "Ran 7 tests in 0.003s",
      "",
      "OK"
    ]
  },
  "code_sha256": {
    "expo

In [4]:
print(pd.DataFrame({"hour": params["hourly_arrivals"]["hours"], "count": params["hourly_arrivals"]["counts"], "probability": params["hourly_arrivals"]["proportions"]}).to_string(index=False))
print(pd.DataFrame({"triage_code": params["triage"]["codes"], "count": params["triage"]["counts"], "probability": params["triage"]["proportions"]}).to_string(index=False))
print(json.dumps(params["unavailable_parameters"], indent=2))

 hour  count  probability
    0   1428     0.025003
    1   1200     0.021011
    2    997     0.017457
    3    926     0.016213
    4    800     0.014007
    5    845     0.014795
    6   1040     0.018210
    7   1460     0.025563
    8   2289     0.040078
    9   2959     0.051810
   10   3400     0.059531
   11   3488     0.061072
   12   3435     0.060144
   13   3454     0.060477
   14   3288     0.057570
   15   3229     0.056537
   16   3264     0.057150
   17   3331     0.058323
   18   3248     0.056870
   19   3222     0.056414
   20   3038     0.053193
   21   2636     0.046154
   22   2362     0.041357
   23   1774     0.031061
 triage_code  count  probability
           1    846     0.014555
           2   8597     0.147908
           3  29568     0.508706
           4  16715     0.287575
           5   2398     0.041257
{
  "provider_service_times": {
    "value": null,
    "status": "unavailable_from_source",
    "owner": "Simulation Team",
    "reason": "No verified s

## Handoff limits
These are pooled unweighted sample distributions, not Houston arrival rates. The Simulation Team must separately provide baseline volume, service times, admission, boarding and capacity assumptions. Weekday labels remain unverified. The compatibility objects do not authorize replacement of the existing PATWT-weighted baseline. Use ED_SCENARIO_PARAMS only after team acceptance. No remote execution is implied by an unexecuted notebook.